# Telco Customer Churn Analysis

**Objective:** Explore customer churn patterns and compare Logistic Regression with Random Forest.

**Dataset required:** `WA_Fn-UseC_-Telco-Customer-Churn.csv`  
Keep the CSV file in the same folder as this notebook before running it.

## 1. Import libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report
)

RANDOM_STATE = 42

## 2. Load and inspect the data

In [ ]:
df = pd.read_csv("WA_Fn-UseC_-Telco-Customer-Churn.csv")

print("Dataset shape:", df.shape)
display(df.head())

In [ ]:
print("Data types:")
display(df.dtypes)

print("Missing values before cleaning:")
display(df.isnull().sum())

## 3. Clean and prepare the data

In [ ]:
# Convert TotalCharges to numeric; blank or invalid values become NaN.
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")

# Fill missing TotalCharges with the median.
df["TotalCharges"] = df["TotalCharges"].fillna(df["TotalCharges"].median())

# Remove the customer identifier because it is not a useful predictive feature.
df = df.drop(columns=["customerID"], errors="ignore")

# Encode the target: No = 0, Yes = 1.
df["Churn"] = df["Churn"].map({"No": 0, "Yes": 1})

# One-hot encode categorical features.
df = pd.get_dummies(df, drop_first=True, dtype=int)

print("Shape after preprocessing:", df.shape)
print("Remaining missing values:", df.isnull().sum().sum())
display(df.head())

## 4. Exploratory Data Analysis (EDA)

In [ ]:
churn_percentage = df["Churn"].mean() * 100
print(f"Percentage of customers who churned: {churn_percentage:.2f}%")

print("\nAverage MonthlyCharges by churn status (0 = No, 1 = Yes):")
display(df.groupby("Churn")["MonthlyCharges"].mean())

print("\nAverage tenure by churn status (0 = No, 1 = Yes):")
display(df.groupby("Churn")["tenure"].mean())

In [ ]:
churn_counts = df["Churn"].value_counts().sort_index()
churn_counts.index = ["No Churn", "Churn"]

ax = churn_counts.plot(kind="bar", figsize=(6, 4))
ax.set_title("Customer Churn Distribution")
ax.set_xlabel("Customer Status")
ax.set_ylabel("Number of Customers")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

## 5. Split the data

In [ ]:
X = df.drop(columns=["Churn"])
y = df["Churn"]

# Use the same test set for both models so the comparison is fair.
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y
)

print("Training rows:", X_train.shape[0])
print("Testing rows:", X_test.shape[0])

## 6. Train and evaluate Logistic Regression

In [ ]:
lr_model = LogisticRegression(max_iter=2000, solver="liblinear", random_state=RANDOM_STATE)
lr_model.fit(X_train, y_train)

lr_pred = lr_model.predict(X_test)

print("Logistic Regression")
print(f"Accuracy : {accuracy_score(y_test, lr_pred):.4f}")
print(f"Precision: {precision_score(y_test, lr_pred, zero_division=0):.4f}")
print(f"Recall   : {recall_score(y_test, lr_pred, zero_division=0):.4f}")
print(f"F1-score : {f1_score(y_test, lr_pred, zero_division=0):.4f}")

print("\nClassification report:")
print(classification_report(
    y_test, lr_pred,
    labels=[0, 1],
    target_names=["Churn = No", "Churn = Yes"],
    zero_division=0
))

In [ ]:
lr_cm = confusion_matrix(y_test, lr_pred, labels=[0, 1])
sns.heatmap(
    lr_cm, annot=True, fmt="d", cmap="Blues",
    xticklabels=["No Churn", "Churn"],
    yticklabels=["No Churn", "Churn"]
)
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Logistic Regression Confusion Matrix")
plt.tight_layout()
plt.show()

## 7. Train and evaluate Random Forest

In [ ]:
rf_model = RandomForestClassifier(
    n_estimators=100,
    random_state=RANDOM_STATE,
    class_weight=None
)
rf_model.fit(X_train, y_train)

rf_pred = rf_model.predict(X_test)

print("Random Forest")
print(f"Accuracy : {accuracy_score(y_test, rf_pred):.4f}")
print(f"Precision: {precision_score(y_test, rf_pred, zero_division=0):.4f}")
print(f"Recall   : {recall_score(y_test, rf_pred, zero_division=0):.4f}")
print(f"F1-score : {f1_score(y_test, rf_pred, zero_division=0):.4f}")

print("\nClassification report:")
print(classification_report(
    y_test, rf_pred,
    labels=[0, 1],
    target_names=["Churn = No", "Churn = Yes"],
    zero_division=0
))

In [ ]:
rf_cm = confusion_matrix(y_test, rf_pred, labels=[0, 1])
sns.heatmap(
    rf_cm, annot=True, fmt="d", cmap="Greens",
    xticklabels=["No Churn", "Churn"],
    yticklabels=["No Churn", "Churn"]
)
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Random Forest Confusion Matrix")
plt.tight_layout()
plt.show()

## 8. Compare both models

In [ ]:
comparison = pd.DataFrame({
    "Model": ["Logistic Regression", "Random Forest"],
    "Accuracy": [
        accuracy_score(y_test, lr_pred),
        accuracy_score(y_test, rf_pred)
    ],
    "Precision": [
        precision_score(y_test, lr_pred, zero_division=0),
        precision_score(y_test, rf_pred, zero_division=0)
    ],
    "Recall (Churn = Yes)": [
        recall_score(y_test, lr_pred, zero_division=0),
        recall_score(y_test, rf_pred, zero_division=0)
    ],
    "F1-score": [
        f1_score(y_test, lr_pred, zero_division=0),
        f1_score(y_test, rf_pred, zero_division=0)
    ]
})

display(comparison.round(4))

lr_recall = recall_score(y_test, lr_pred, zero_division=0)
rf_recall = recall_score(y_test, rf_pred, zero_division=0)

if lr_recall > rf_recall:
    print("Logistic Regression has higher recall for Churn = Yes.")
elif rf_recall > lr_recall:
    print("Random Forest has higher recall for Churn = Yes.")
else:
    print("Both models have the same recall for Churn = Yes.")

## 9. Random Forest feature importance

In [ ]:
rf_importance = pd.DataFrame({
    "Feature": X.columns,
    "Importance": rf_model.feature_importances_
}).sort_values("Importance", ascending=False)

print("Top 10 features:")
display(rf_importance.head(10))

top10 = rf_importance.head(10).sort_values("Importance")
plt.figure(figsize=(9, 6))
plt.barh(top10["Feature"], top10["Importance"])
plt.xlabel("Feature Importance")
plt.title("Top 10 Random Forest Features")
plt.tight_layout()
plt.show()

## 10. Interpret the confusion matrix

In [ ]:
# For the positive class (Churn = Yes):
# TN = correctly predicted No Churn
# FP = predicted Churn, but actual status was No Churn
# FN = actual Churn, but predicted No Churn
# TP = correctly predicted Churn

TN, FP, FN, TP = confusion_matrix(y_test, rf_pred, labels=[0, 1]).ravel()

print("Correctly predicted as churned (TP):", TP)
print("Incorrectly predicted as churned (FP):", FP)
print("Actually churned but predicted as non-churned (FN):", FN)
print("Correctly predicted as non-churned (TN):", TN)

print("\nTop 3 features associated with Random Forest predictions:")
display(rf_importance.head(3))
print(
    "\nNote: Feature importance indicates how useful a feature was to the model's "
    "predictions; it does not prove that the feature causes customer churn."
)

## 11. Key takeaways

After running all cells, use the actual metric table and feature-importance results to write 2–3 findings. For example, mention which model has higher recall for `Churn = Yes` and which customer features rank highest.

**Reproducibility note:** Both models use the same stratified 80/20 train-test split. This makes the evaluation comparison consistent.